# GPTQ From Scratch â€” Multi-Architecture Benchmark (A100)

Implementation complete de GPTQ (Frantar et al., 2022) from scratch, testee sur 3 architectures :

| Modele | Params | Architecture |
|--------|--------|--------------|
| `gpt2-xl` | 1.5B | GPT-2 (Conv1D) |
| `facebook/opt-1.3b` | 1.3B | OPT (Linear) |
| `meta-llama/Meta-Llama-3-8B` | 8B | LLaMA (Linear + RoPE) |

**Optimisations implementees :**
- **Grouping** (`group_size=128`) : scales par groupe de colonnes au lieu de par ligne
- **Act-order** : quantifie les colonnes par magnitude d'activation decroissante
- **True-sequential** : quantifie les sous-couches sequentiellement dans chaque bloc

**Setup :** Runtime > Change runtime type > **A100 GPU**

Llama-3 necessite un token HuggingFace avec acces au modele.

## 0. Setup

In [ ]:
!pip install -q torch transformers>=4.40.0 accelerate>=0.27.0 datasets tqdm

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU required â€” change runtime type"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 35:
    print("\nAttention : <40 GB VRAM â€” Llama-3-8B risque un OOM.")
    print("GPT-2-XL et OPT-1.3b fonctionneront sans probleme.")

In [ ]:
import getpass, os

HF_TOKEN = getpass.getpass("HuggingFace token (Enter pour skip Llama-3) : ")
if HF_TOKEN.strip():
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("Token OK.")
else:
    HF_TOKEN = None
    print("Pas de token â€” Llama-3 sera skip.")

## 1. Source code

Les cellules suivantes ecrivent les 5 fichiers source sur disque via `%%writefile`.

**Executez-les toutes dans l'ordre avant de lancer le benchmark.**

In [ ]:
%%writefile quantize.py
"""
Quantization and dequantization utilities.
Symmetric uniform quantization to n-bit integers.
"""

import torch


def quantize_tensor(w, n_bits=4):
    qmax = 2 ** (n_bits - 1) - 1
    qmin = -(2 ** (n_bits - 1))
    scale = w.abs().max() / qmax
    scale = scale.clamp(min=1e-10)
    q = torch.clamp(torch.round(w / scale), qmin, qmax)
    return q * scale, scale


def compute_row_scales(W, n_bits=4):
    qmax = 2 ** (n_bits - 1) - 1
    scales = W.abs().amax(dim=1) / qmax
    return scales.clamp(min=1e-10)


def quantize_column(w_col, scales, n_bits=4):
    qmax = 2 ** (n_bits - 1) - 1
    qmin = -(2 ** (n_bits - 1))
    q = torch.clamp(torch.round(w_col / scales), qmin, qmax)
    return q * scales


def round_to_nearest(w, n_bits=4):
    w_hat, _ = quantize_tensor(w, n_bits)
    return w_hat

In [ ]:
%%writefile arch_config.py
"""
Architecture-specific configuration for multi-model GPTQ support.
Supports GPT-2, LLaMA (Llama-2/3 + Mistral), and OPT.
"""

from dataclasses import dataclass, field
from typing import Callable, List, Optional

import torch


@dataclass
class ArchConfig:
    get_blocks: Callable
    get_final_ln: Callable
    compute_embeddings: Callable
    block_forward: Callable
    get_block_kwargs: Callable
    get_max_seq_len: Callable
    layer_name_prefix: str
    sublayer_groups: Optional[List[List[str]]] = field(default=None)  # for true-sequential


# ---- GPT-2 ----
def _gpt2_get_blocks(model):       return model.transformer.h
def _gpt2_get_final_ln(model):     return model.transformer.ln_f
def _gpt2_compute_embeddings(model, input_ids, device):
    t = model.transformer
    pos = torch.arange(input_ids.shape[1], device=device).unsqueeze(0)
    return t.wte(input_ids) + t.wpe(pos)
def _gpt2_block_forward(block, hidden, **kw): return block(hidden, **kw)[0]
def _gpt2_get_block_kwargs(model, input_ids, device): return {}
def _gpt2_get_max_seq_len(model):  return model.config.n_positions

GPT2_CONFIG = ArchConfig(
    _gpt2_get_blocks, _gpt2_get_final_ln, _gpt2_compute_embeddings,
    _gpt2_block_forward, _gpt2_get_block_kwargs, _gpt2_get_max_seq_len,
    "transformer.h",
    sublayer_groups=[
        ["attn.c_attn"],
        ["attn.c_proj"],
        ["mlp.c_fc"],
        ["mlp.c_proj"],
    ],
)


# ---- LLaMA (Llama-2/3, Mistral) ----
def _llama_get_blocks(model):      return model.model.layers
def _llama_get_final_ln(model):    return model.model.norm
def _llama_compute_embeddings(model, input_ids, device):
    return model.model.embed_tokens(input_ids)
def _llama_block_forward(block, hidden, **kw):
    # transformers >= 4.44 returns a tensor; older versions return a tuple
    out = block(hidden, **kw)
    return out[0] if isinstance(out, tuple) else out
def _llama_get_block_kwargs(model, input_ids, device):
    seq_len = input_ids.shape[1]
    position_ids = torch.arange(0, seq_len, device=device).unsqueeze(0)
    kwargs = {"position_ids": position_ids}
    # position_embeddings=(cos, sin) is mandatory in transformers >= 4.44
    rotary_emb = getattr(model.model, "rotary_emb", None)
    if rotary_emb is not None:
        dtype = next(model.parameters()).dtype
        dummy = torch.empty(1, seq_len, 1, device=device, dtype=dtype)
        cos, sin = rotary_emb(dummy, position_ids)
        kwargs["position_embeddings"] = (cos, sin)
    return kwargs
def _llama_get_max_seq_len(model):
    return getattr(model.config, "max_position_embeddings", 4096)

LLAMA_CONFIG = ArchConfig(
    _llama_get_blocks, _llama_get_final_ln, _llama_compute_embeddings,
    _llama_block_forward, _llama_get_block_kwargs, _llama_get_max_seq_len,
    "model.layers",
    sublayer_groups=[
        ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj"],
        ["self_attn.o_proj"],
        ["mlp.gate_proj", "mlp.up_proj"],
        ["mlp.down_proj"],
    ],
)


# ---- OPT ----
def _opt_get_blocks(model):        return model.model.decoder.layers
def _opt_get_final_ln(model):
    d = model.model.decoder
    return d.final_layer_norm if hasattr(d, "final_layer_norm") else None
def _opt_compute_embeddings(model, input_ids, device):
    d = model.model.decoder
    h = d.embed_tokens(input_ids) + d.embed_positions(torch.ones_like(input_ids))
    if hasattr(d, "project_in") and d.project_in is not None:
        h = d.project_in(h)
    return h
def _opt_block_forward(block, hidden, **kw): return block(hidden, **kw)[0]
def _opt_get_block_kwargs(model, input_ids, device): return {}
def _opt_get_max_seq_len(model):
    return getattr(model.config, "max_position_embeddings", 2048)

OPT_CONFIG = ArchConfig(
    _opt_get_blocks, _opt_get_final_ln, _opt_compute_embeddings,
    _opt_block_forward, _opt_get_block_kwargs, _opt_get_max_seq_len,
    "model.decoder.layers",
    sublayer_groups=[
        ["self_attn.q_proj", "self_attn.k_proj", "self_attn.v_proj"],
        ["self_attn.out_proj"],
        ["fc1"],
        ["fc2"],
    ],
)


# ---- Auto-detection ----
_CONFIG_CLASS_MAP = {
    "GPT2Config": GPT2_CONFIG,
    "LlamaConfig": LLAMA_CONFIG,
    "MistralConfig": LLAMA_CONFIG,
    "OPTConfig": OPT_CONFIG,
}

def get_arch_config(model) -> ArchConfig:
    name = model.config.__class__.__name__
    if name in _CONFIG_CLASS_MAP:
        return _CONFIG_CLASS_MAP[name]
    raise ValueError(f"Unsupported: {name}. Supported: {list(_CONFIG_CLASS_MAP)}")

In [ ]:
%%writefile model_utils.py
"""
Model utilities: loading, calibration data, block-wise processing.
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from arch_config import get_arch_config


def load_model(model_name="gpt2", device="cuda", token=None):
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=dtype, token=token,
    )
    model = model.to(device)
    model.eval()
    return model, tokenizer


def get_transformer_blocks(model):
    arch = get_arch_config(model)
    return arch, arch.get_blocks(model)


def get_calibration_data(tokenizer, n_samples=128, seq_len=2048, seed=42,
                         dataset_name="wikitext2"):
    if dataset_name == "c4":
        dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)
        samples = []
        for doc in dataset:
            toks = tokenizer.encode(doc["text"])
            if len(toks) >= seq_len:
                samples.append(torch.tensor(toks[:seq_len], dtype=torch.long).unsqueeze(0))
                if len(samples) >= n_samples:
                    break
        return samples
    else:
        dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        text = "\n\n".join(dataset["text"])
        tokens = tokenizer.encode(text)
        tokens = torch.tensor(tokens, dtype=torch.long)

    rng = torch.Generator()
    rng.manual_seed(seed)

    samples = []
    for _ in range(n_samples):
        start = torch.randint(0, len(tokens) - seq_len, (1,), generator=rng).item()
        samples.append(tokens[start : start + seq_len].unsqueeze(0))
    return samples


@torch.no_grad()
def get_block_inputs(model, calibration_data, device="cpu"):
    arch = get_arch_config(model)
    return [arch.compute_embeddings(model, b.to(device), device) for b in calibration_data]


def get_weight_and_type(layer):
    if type(layer).__name__ == "Conv1D":
        return layer.weight.data.T.clone(), True
    return layer.weight.data.clone(), False


def set_weight(layer, Q, is_conv1d):
    if is_conv1d:
        layer.weight.data = Q.T.to(layer.weight.dtype)
    else:
        layer.weight.data = Q.to(layer.weight.dtype)

In [ ]:
%%writefile gptq.py
"""
GPTQ algorithm implementation (Frantar et al., 2022).
Block-by-block quantization with Hessian-based error compensation.

Optimizations:
  - Grouping: per-group-of-g-columns scales instead of per-row (group_size=128)
  - Act-order: quantize columns sorted by descending Hessian diagonal
  - True-sequential: quantize sub-layers sequentially within each block
"""

import time
import torch
from tqdm import tqdm
from quantize import compute_row_scales, quantize_column


def compute_hessian(X, damp_pct=0.01):
    n = X.shape[0]
    H = X.T @ X / n
    damp = damp_pct * torch.mean(torch.diag(H))
    diag_idx = range(H.shape[0])
    H[diag_idx, diag_idx] += damp
    return H


def gptq_quantize_layer(W, H, n_bits=4, block_size=128, group_size=-1, act_order=False):
    W = W.clone().float()
    n_rows, n_cols = W.shape

    # Act-order: permute columns by descending Hessian diagonal
    perm = None
    if act_order:
        perm = torch.argsort(torch.diag(H), descending=True)
        W = W[:, perm]
        H = H[perm][:, perm]

    try:
        L = torch.linalg.cholesky(H)
        H_inv = torch.cholesky_inverse(L)
    except RuntimeError:
        extra = 0.1 * torch.mean(torch.diag(H))
        H_reg = H.clone()
        H_reg[range(H.shape[0]), range(H.shape[0])] += extra
        L = torch.linalg.cholesky(H_reg)
        H_inv = torch.cholesky_inverse(L)

    # Scales: per-group (recomputed at group boundaries) or per-row (fixed)
    if group_size > 0:
        current_group = -1
        scales = None
    else:
        scales = compute_row_scales(W, n_bits)

    Q = torch.zeros_like(W)
    loss = 0.0

    for i1 in range(0, n_cols, block_size):
        i2 = min(i1 + block_size, n_cols)
        count = i2 - i1
        W_blk = W[:, i1:i2].clone()
        Q_blk = torch.zeros_like(W_blk)
        Err = torch.zeros_like(W_blk)
        H_inv_blk = H_inv[i1:i2, i1:i2]

        for j in range(count):
            col_idx = i1 + j

            # Recompute scales at group boundaries
            if group_size > 0:
                g = col_idx // group_size
                if g != current_group:
                    current_group = g
                    g_start = g * group_size
                    g_end = min(g_start + group_size, n_cols)
                    scales = compute_row_scales(W[:, g_start:g_end], n_bits)

            w_col = W_blk[:, j]
            h_jj = H_inv_blk[j, j]
            q_col = quantize_column(w_col, scales, n_bits)
            Q_blk[:, j] = q_col
            err = (w_col - q_col) / h_jj
            Err[:, j] = err
            loss += ((w_col - q_col) ** 2 / h_jj).sum().item()
            if j < count - 1:
                W_blk[:, j+1:] -= err.unsqueeze(1) * H_inv_blk[j, j+1:].unsqueeze(0)

        Q[:, i1:i2] = Q_blk
        if i2 < n_cols:
            W[:, i2:] -= Err @ H_inv[i1:i2, i2:]

    # Inverse permute if act-order was used
    if perm is not None:
        inv_perm = torch.argsort(perm)
        Q = Q[:, inv_perm]

    return Q, loss


def _capture_and_quantize(block, layers, hidden_states, arch, block_kwargs,
                          block_idx, device, n_bits, block_size, group_size, act_order):
    """Hook layers, run forward passes to capture Hessians, then quantize."""
    from model_utils import get_weight_and_type, set_weight

    layer_H = {name: None for name, _ in layers}
    layer_n = {name: 0 for name, _ in layers}

    hooks = []
    for name, module in layers:
        def make_hook(ln):
            def hook_fn(mod, inp, out):
                x = inp[0].detach().reshape(-1, inp[0].shape[-1]).float()
                H = x.T @ x
                if layer_H[ln] is None:
                    layer_H[ln] = H
                else:
                    layer_H[ln] += H
                layer_n[ln] += x.shape[0]
            return hook_fn
        hooks.append(module.register_forward_hook(make_hook(name)))

    for h in hidden_states:
        arch.block_forward(block, h.to(device), **block_kwargs)
    for handle in hooks:
        handle.remove()

    stats = {}
    for name, module in layers:
        t0 = time.time()
        full_name = f"{arch.layer_name_prefix}.{block_idx}.{name}"

        H = layer_H[name] / layer_n[name]
        damp = 0.01 * torch.mean(torch.diag(H))
        H[range(H.shape[0]), range(H.shape[0])] += damp

        W, is_conv1d = get_weight_and_type(module)
        Q, loss = gptq_quantize_layer(
            W.float(), H, n_bits=n_bits, block_size=block_size,
            group_size=group_size, act_order=act_order,
        )
        set_weight(module, Q, is_conv1d)
        stats[full_name] = {"loss": loss, "time": time.time() - t0}
        del H, W, Q
        layer_H[name] = None

    del layer_H, layer_n
    torch.cuda.empty_cache()
    return stats


@torch.no_grad()
def quantize_model(model, calibration_data, n_bits=4, block_size=128,
                   group_size=-1, act_order=False, true_sequential=False,
                   device="cpu"):
    from model_utils import get_transformer_blocks, get_block_inputs

    arch, blocks = get_transformer_blocks(model)
    sample_ids = calibration_data[0]
    block_kwargs = arch.get_block_kwargs(model, sample_ids, device)

    print("Computing embedding outputs for calibration data...")
    hidden_states = get_block_inputs(model, calibration_data, device=device)
    hidden_states = [h.cpu() for h in hidden_states]
    torch.cuda.empty_cache()
    print(f"Got {len(hidden_states)} calibration samples")

    stats = {}

    for block_idx, block in enumerate(tqdm(blocks, desc=f"GPTQ {n_bits}-bit")):
        layers_to_quantize = []
        for name, module in block.named_modules():
            if isinstance(module, torch.nn.Linear) or type(module).__name__ == "Conv1D":
                layers_to_quantize.append((name, module))

        if true_sequential and arch.sublayer_groups is not None:
            # Process sub-layer groups sequentially, re-capturing activations each time
            quantized_names = set()
            for group_names in arch.sublayer_groups:
                layers_in_group = [(n, m) for n, m in layers_to_quantize if n in group_names]
                if not layers_in_group:
                    continue
                group_stats = _capture_and_quantize(
                    block, layers_in_group, hidden_states, arch, block_kwargs,
                    block_idx, device, n_bits, block_size, group_size, act_order,
                )
                stats.update(group_stats)
                quantized_names.update(n for n, _ in layers_in_group)

            remaining = [(n, m) for n, m in layers_to_quantize if n not in quantized_names]
            if remaining:
                rem_stats = _capture_and_quantize(
                    block, remaining, hidden_states, arch, block_kwargs,
                    block_idx, device, n_bits, block_size, group_size, act_order,
                )
                stats.update(rem_stats)
        else:
            block_stats = _capture_and_quantize(
                block, layers_to_quantize, hidden_states, arch, block_kwargs,
                block_idx, device, n_bits, block_size, group_size, act_order,
            )
            stats.update(block_stats)

        new_hidden = []
        for h in hidden_states:
            new_hidden.append(arch.block_forward(block, h.to(device), **block_kwargs).detach().cpu())
        hidden_states = new_hidden

    return stats

In [ ]:
%%writefile evaluate.py
"""
Perplexity evaluation on WikiText-2 test set.
"""

import torch
from datasets import load_dataset
from tqdm import tqdm
from arch_config import get_arch_config


@torch.no_grad()
def evaluate_perplexity(model, tokenizer, device="cuda", stride=512, max_len=None):
    """
    Evaluate perplexity on WikiText-2 test set using sliding window.

    Args:
        max_len: Context window size override (default: model's max).
    """
    arch = get_arch_config(model)

    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n\n".join(dataset["text"])
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(device)

    if max_len is None:
        max_len = arch.get_max_seq_len(model)
    total_len = input_ids.size(1)

    nlls = []
    n_tokens = 0

    for begin in tqdm(range(0, total_len, stride), desc="Evaluating perplexity"):
        end = min(begin + max_len, total_len)
        input_chunk = input_ids[:, begin:end]
        labels = input_chunk.clone()
        n_ctx = 0 if begin == 0 else max_len - stride
        labels[:, :n_ctx] = -100

        outputs = model(input_chunk, labels=labels)

        n_valid = (labels != -100).sum().item()
        if n_valid > 0:
            nlls.append(outputs.loss.float() * n_valid)
            n_tokens += n_valid

        if end == total_len:
            break

    avg_nll = torch.stack(nlls).sum() / n_tokens
    return torch.exp(avg_nll).item()

## 2. Sanity check â€” GPT-2 small (124M)

Verification rapide que tout fonctionne avant de lancer les gros modeles.
GPT-2 4-bit GPTQ devrait donner ~34.5 de perplexite.

In [ ]:
from model_utils import load_model, get_calibration_data
from arch_config import get_arch_config
from gptq import quantize_model
from evaluate import evaluate_perplexity
import time, gc

print("--- Sanity check: gpt2 (124M) ---")
model, tok = load_model("gpt2", device="cuda")
arch = get_arch_config(model)
print(f"Architecture: {model.config.__class__.__name__}")

# Baseline
ppl_base = evaluate_perplexity(model, tok, device="cuda")
print(f"Baseline PPL: {ppl_base:.2f}")

# GPTQ 4-bit
seq_len = min(arch.get_max_seq_len(model), 2048)
calib = get_calibration_data(tok, n_samples=128, seq_len=seq_len)
stats = quantize_model(model, calib, n_bits=4, device="cuda")
ppl_q = evaluate_perplexity(model, tok, device="cuda")
print(f"GPTQ 4-bit PPL: {ppl_q:.2f} (delta: +{ppl_q - ppl_base:.2f})")
print(f"Layers quantized: {len(stats)}")

del model, tok, calib, stats
gc.collect()
torch.cuda.empty_cache()
print("Sanity check OK.")

## 3. Full benchmark â€” GPT-2-XL, OPT-1.3b, Llama-3-8B

Chaque modele est charge, evalue en baseline, quantize en GPTQ 4-bit, puis re-evalue.

En cas d'OOM, le modele est skip et on passe au suivant.

In [ ]:
def vram_used():
    """Current GPU memory usage in GB."""
    return torch.cuda.memory_allocated() / 1e9

def vram_total():
    return torch.cuda.get_device_properties(0).total_memory / 1e9


def run_benchmark(model_name, token=None, n_bits=4, n_samples=128,
                  block_size=128, eval_max_len=2048,
                  group_size=-1, act_order=False, true_sequential=False,
                  calib_dataset="wikitext2"):
    """
    Full GPTQ benchmark on a single model.

    eval_max_len caps the context window during perplexity evaluation
    (Llama-3 has 8192 natively, which is slow and VRAM-hungry).
    """
    device = "cuda"
    res = {"model": model_name, "bits": n_bits,
           "group_size": group_size, "act_order": act_order,
           "true_sequential": true_sequential,
           "calib_dataset": calib_dataset}

    tricks = []
    if group_size > 0:
        tricks.append(f"g={group_size}")
    if act_order:
        tricks.append("act-order")
    if true_sequential:
        tricks.append("true-seq")
    trick_str = f" [{', '.join(tricks)}]" if tricks else ""

    print(f"\n{'='*65}")
    print(f" {model_name} â€” {n_bits}-bit{trick_str} (calib: {calib_dataset})")
    print(f"{'='*65}")

    # Load
    model, tokenizer = load_model(model_name, device=device, token=token)
    arch = get_arch_config(model)
    n_params = sum(p.numel() for p in model.parameters())
    res["params_M"] = round(n_params / 1e6, 1)
    res["arch"] = model.config.__class__.__name__

    model_max = arch.get_max_seq_len(model)
    eff_eval_len = min(model_max, eval_max_len)

    print(f"  Arch       : {res['arch']}")
    print(f"  Params     : {res['params_M']}M")
    print(f"  Max seq    : {model_max} (eval capped at {eff_eval_len})")
    print(f"  VRAM used  : {vram_used():.1f} / {vram_total():.0f} GB")

    # Baseline perplexity
    print("\n  [1/3] Baseline perplexity...")
    t0 = time.time()
    ppl_base = evaluate_perplexity(model, tokenizer, device=device,
                                   stride=512, max_len=eff_eval_len)
    res["baseline_ppl"] = round(ppl_base, 2)
    res["baseline_time"] = round(time.time() - t0, 1)
    print(f"        PPL = {res['baseline_ppl']}  ({res['baseline_time']}s)")

    # Calibration + GPTQ
    calib_seq_len = min(model_max, 2048)
    print(f"\n  [2/3] GPTQ {n_bits}-bit{trick_str}  (calib: {calib_dataset}, {n_samples} x {calib_seq_len})...")
    calib = get_calibration_data(tokenizer, n_samples=n_samples,
                                 seq_len=calib_seq_len,
                                 dataset_name=calib_dataset)
    t0 = time.time()
    stats = quantize_model(model, calib, n_bits=n_bits,
                           block_size=block_size,
                           group_size=group_size,
                           act_order=act_order,
                           true_sequential=true_sequential,
                           device=device)
    res["quant_time"] = round(time.time() - t0, 1)
    res["n_layers"] = len(stats)
    res["total_loss"] = round(sum(s["loss"] for s in stats.values()), 1)
    print(f"        {res['n_layers']} layers in {res['quant_time']}s")

    # Post-quant perplexity
    print(f"\n  [3/3] Post-quantization perplexity...")
    t0 = time.time()
    ppl_q = evaluate_perplexity(model, tokenizer, device=device,
                                stride=512, max_len=eff_eval_len)
    res["quant_ppl"] = round(ppl_q, 2)
    res["eval_time"] = round(time.time() - t0, 1)
    res["delta_ppl"] = round(ppl_q - ppl_base, 2)
    print(f"        PPL = {res['quant_ppl']}  (delta: +{res['delta_ppl']})  ({res['eval_time']}s)")
    print(f"        VRAM peak: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

    # Cleanup
    del model, tokenizer, calib, stats
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    return res

In [ ]:
# Models to benchmark (ascending size)
models = [
    ("gpt2-xl",                          None),       # 1.5B â€” GPT-2
    ("facebook/opt-1.3b",                None),       # 1.3B â€” OPT
]

if HF_TOKEN:
    models.append(
        ("meta-llama/Meta-Llama-3-8B",   HF_TOKEN),  # 8B â€” LLaMA
    )
else:
    print("Pas de HF token â€” Llama-3-8B sera skip.")

all_results = []

for model_name, token in models:
    try:
        r = run_benchmark(model_name, token=token)
        all_results.append(r)
    except torch.cuda.OutOfMemoryError:
        print(f"\n  OOM sur {model_name} â€” skip.")
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"\n  Erreur sur {model_name}: {e} â€” skip.")
        gc.collect()
        torch.cuda.empty_cache()

## 4. Results

In [ ]:
print()
print("=" * 85)
print("  GPTQ 4-bit Quantization Results â€” WikiText-2 Perplexity (vanilla)")
print("=" * 85)
print(f"  {'Model':<35} {'Params':>8} {'FP16':>9} {'GPTQ-4b':>9} {'Delta':>8} {'Time':>8}")
print("-" * 85)
for r in all_results:
    print(
        f"  {r['model']:<35} {r['params_M']:>7.1f}M"
        f" {r['baseline_ppl']:>9.2f} {r['quant_ppl']:>9.2f}"
        f" {r['delta_ppl']:>+7.2f} {r['quant_time']:>7.1f}s"
    )
print("=" * 85)
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print()

## 5. Optimized benchmark â€” grouping + act-order + true-sequential

Memes modeles, cette fois avec les 3 optimisations du papier GPTQ :
- **Grouping** `g=128` : scales par groupe de 128 colonnes
- **Act-order** : colonnes quantifiees par magnitude d'activation decroissante
- **True-sequential** : sous-couches (QKV â†’ O â†’ MLP) quantifiees sequentiellement

L'objectif est de reduire le delta de perplexite par rapport au vanilla GPTQ.

In [ ]:
opt_results = []

for model_name, token in models:
    try:
        r = run_benchmark(model_name, token=token,
                          group_size=128, act_order=True, true_sequential=True)
        opt_results.append(r)
    except torch.cuda.OutOfMemoryError:
        print(f"\n  OOM sur {model_name} â€” skip.")
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"\n  Erreur sur {model_name}: {e} â€” skip.")
        gc.collect()
        torch.cuda.empty_cache()

## 6. Comparison: vanilla vs optimized

In [ ]:
# Build comparison lookup: model_name -> optimized result
opt_lookup = {r["model"]: r for r in opt_results}

print()
print("=" * 95)
print("  GPTQ 4-bit â€” Vanilla vs Optimized (g=128, act-order, true-seq)")
print("=" * 95)
print(f"  {'Model':<35} {'Params':>8} {'FP16':>8} {'Vanilla':>9} {'Optimized':>10} {'Improve':>9}")
print("-" * 95)
for v in all_results:
    name = v["model"]
    o = opt_lookup.get(name)
    if o:
        improve = v["delta_ppl"] - o["delta_ppl"]
        print(
            f"  {name:<35} {v['params_M']:>7.1f}M"
            f" {v['baseline_ppl']:>8.2f}"
            f" {v['quant_ppl']:>8.2f} (+{v['delta_ppl']:.2f})"
            f" {o['quant_ppl']:>8.2f} (+{o['delta_ppl']:.2f})"
            f" {improve:>+8.2f}"
        )
    else:
        print(
            f"  {name:<35} {v['params_M']:>7.1f}M"
            f" {v['baseline_ppl']:>8.2f}"
            f" {v['quant_ppl']:>8.2f} (+{v['delta_ppl']:.2f})"
            f" {'N/A':>19}"
            f" {'':>9}"
        )
print("=" * 95)
print(f"  GPU: {torch.cuda.get_device_name(0)}")
print(f"  Positive 'Improve' = optimized is closer to FP16 baseline")
print()

## 7. C4 calibration benchmark

Le papier GPTQ original utilise C4 comme dataset de calibration.
On relance les memes benchmarks (vanilla + optimized) avec `calib_dataset="c4"` pour comparer.

In [ ]:
# C4 calibration â€” vanilla GPTQ 4-bit
c4_vanilla_results = []

for model_name, token in models:
    try:
        r = run_benchmark(model_name, token=token, calib_dataset="c4")
        c4_vanilla_results.append(r)
    except torch.cuda.OutOfMemoryError:
        print(f"\n  OOM sur {model_name} â€” skip.")
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"\n  Erreur sur {model_name}: {e} â€” skip.")
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
# C4 calibration â€” optimized GPTQ 4-bit (g=128, act-order, true-seq)
c4_opt_results = []

for model_name, token in models:
    try:
        r = run_benchmark(model_name, token=token,
                          group_size=128, act_order=True, true_sequential=True,
                          calib_dataset="c4")
        c4_opt_results.append(r)
    except torch.cuda.OutOfMemoryError:
        print(f"\n  OOM sur {model_name} â€” skip.")
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"\n  Erreur sur {model_name}: {e} â€” skip.")
        gc.collect()
        torch.cuda.empty_cache()

## 8. Comparison: WikiText-2 vs C4 calibration

Tableau final comparant les resultats avec les deux datasets de calibration,
pour chaque modele en vanilla et optimized.

In [ ]:
# Build lookups
w2_vanilla_lookup = {r['model']: r for r in all_results}
w2_opt_lookup = {r['model']: r for r in opt_results}
c4_vanilla_lookup = {r['model']: r for r in c4_vanilla_results}
c4_opt_lookup = {r['model']: r for r in c4_opt_results}

all_model_names = []
for name, _ in models:
    if name not in all_model_names:
        all_model_names.append(name)

print()
print('=' * 110)
print('  GPTQ 4-bit -- WikiText-2 calib vs C4 calib (WikiText-2 PPL evaluation)')
print('=' * 110)
header = '  {:<30} {:>7}  {:>7}  {:>14} {:>14}  {:>14} {:>14}'.format(
    'Model', 'Params', 'FP16', 'W2 Vanilla', 'W2 Optimized', 'C4 Vanilla', 'C4 Optimized')
print(header)
print('-' * 110)

for name in all_model_names:
    w2v = w2_vanilla_lookup.get(name)
    w2o = w2_opt_lookup.get(name)
    c4v = c4_vanilla_lookup.get(name)
    c4o = c4_opt_lookup.get(name)

    if not w2v:
        continue

    params = f"{w2v['params_M']:.0f}M"
    fp16 = f"{w2v['baseline_ppl']:.2f}"

    def fmt(r):
        if r is None:
            return 'N/A'
        return f"{r['quant_ppl']:.2f} (+{r['delta_ppl']:.2f})"

    row = '  {:<30} {:>7}  {:>7}  {:>14} {:>14}  {:>14} {:>14}'.format(
        name, params, fp16, fmt(w2v), fmt(w2o), fmt(c4v), fmt(c4o))
    print(row)

print('=' * 110)
print(f'  GPU: {torch.cuda.get_device_name(0)}')
print('  Evaluation: WikiText-2 test set | Calibration: WikiText-2 train (W2) vs C4 train')
print()


## 9. Generate README figures

Generate publication-quality figures from the benchmark results collected above.
Figures are saved to `figures/` and displayed inline.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os

os.makedirs("figures", exist_ok=True)

# Style
C_GREY = "#aaaaaa"
C_ORANGE = "#e67e22"
C_BLUE = "#2980b9"
C_RED = "#e74c3c"
C_GREEN = "#27ae60"

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.15,
})

def short_name(name):
    mapping = {
        "gpt2": "GPT-2\n(124M)",
        "gpt2-xl": "GPT-2-XL\n(1.6B)",
        "facebook/opt-1.3b": "OPT-1.3B\n(1.4B)",
        "meta-llama/Meta-Llama-3-8B": "Llama-3-8B\n(8B)",
    }
    return mapping.get(name, name)

generated = []

# =====================================================================
# 1. results_vanilla.png
# =====================================================================
if all_results:
    names = [short_name(r['model']) for r in all_results]
    fp16 = [r['baseline_ppl'] for r in all_results]
    gptq = [r['quant_ppl'] for r in all_results]

    x = np.arange(len(names))
    w = 0.32

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(x - w/2, fp16, w, color=C_GREY, label="FP16 baseline", edgecolor="white")
    ax.bar(x + w/2, gptq, w, color=C_BLUE, label="GPTQ 4-bit", edgecolor="white")

    for i in range(len(names)):
        delta_pct = (gptq[i] - fp16[i]) / fp16[i] * 100
        ax.text(x[i] + w/2, gptq[i] + 0.15, f"+{delta_pct:.1f}%",
                ha="center", va="bottom", fontsize=10, color=C_BLUE, fontweight="bold")

    ax.set_ylabel("WikiText-2 Perplexity (lower is better)", fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.set_ylim(0, max(gptq) * 1.2)
    ax.legend(loc="upper right", framealpha=0.9, fontsize=11)
    ax.set_title("GPTQ 4-bit Quantization -- WikiText-2 Perplexity",
                 fontsize=14, fontweight="bold")
    ax.yaxis.grid(True, alpha=0.3)

    fig.savefig("figures/results_vanilla.png")
    plt.show()
    plt.close(fig)
    generated.append("results_vanilla.png")

# =====================================================================
# 2. results_vanilla_vs_optimized.png
# =====================================================================
if opt_results:
    opt_lk = {r['model']: r for r in opt_results}
    paired = [(r, opt_lk[r['model']]) for r in all_results if r['model'] in opt_lk]

    if paired:
        names = [short_name(v['model']) for v, _ in paired]
        fp16 = [v['baseline_ppl'] for v, _ in paired]
        vanilla = [v['quant_ppl'] for v, _ in paired]
        optimized = [o['quant_ppl'] for _, o in paired]

        x = np.arange(len(names))
        w = 0.25

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.bar(x - w, fp16, w, color=C_GREY, label="FP16 baseline", edgecolor="white")
        ax.bar(x, vanilla, w, color=C_ORANGE, label="Vanilla GPTQ 4-bit", edgecolor="white")
        ax.bar(x + w, optimized, w, color=C_BLUE, label="Optimized GPTQ 4-bit", edgecolor="white")

        for i in range(len(names)):
            dv = vanilla[i] - fp16[i]
            do = optimized[i] - fp16[i]
            ax.text(x[i], vanilla[i] + 0.15, f"+{dv:.2f}",
                    ha="center", va="bottom", fontsize=9, color=C_ORANGE, fontweight="bold")
            ax.text(x[i] + w, optimized[i] + 0.15, f"+{do:.2f}",
                    ha="center", va="bottom", fontsize=9, color=C_BLUE, fontweight="bold")

        ax.set_ylabel("WikiText-2 Perplexity (lower is better)", fontsize=12)
        ax.set_xticks(x)
        ax.set_xticklabels(names)
        ax.set_ylim(0, max(vanilla) + 2.5)
        ax.legend(loc="upper right", framealpha=0.9, fontsize=11)
        ax.set_title("Vanilla vs Optimized GPTQ 4-bit (g=128 + act-order + true-seq)",
                     fontsize=14, fontweight="bold")
        ax.yaxis.grid(True, alpha=0.3)

        fig.savefig("figures/results_vanilla_vs_optimized.png")
        plt.show()
        plt.close(fig)
        generated.append("results_vanilla_vs_optimized.png")

# =====================================================================
# 3. gptq_vs_rtn.png
# =====================================================================
# Use sanity check results if available, else known values
try:
    _gpt2_fp16 = ppl_base
    _gpt2_gptq = ppl_q
except NameError:
    _gpt2_fp16 = 25.17
    _gpt2_gptq = 35.45

_gpt2_rtn = 14434  # known from prior runs

labels = ["FP16\nbaseline", "GPTQ\n4-bit", "Naive RTN\n4-bit"]
values = [_gpt2_fp16, _gpt2_gptq, _gpt2_rtn]
colors = [C_GREY, C_BLUE, C_RED]

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(labels, values, color=colors, edgecolor="white", width=0.55)

ax.set_yscale("log")
ax.set_ylabel("WikiText-2 Perplexity (log scale)", fontsize=12)
ax.set_title("Why GPTQ Matters: GPTQ vs Naive RTN on GPT-2",
             fontsize=14, fontweight="bold")
ax.yaxis.grid(True, alpha=0.3)

for bar, val in zip(bars, values):
    y = bar.get_height()
    fmt = f"{val:,.0f}" if val > 100 else f"{val:.2f}"
    ax.text(bar.get_x() + bar.get_width() / 2, y * 1.15, fmt,
            ha="center", va="bottom", fontsize=11, fontweight="bold")

ratio = round(_gpt2_rtn / _gpt2_gptq)
ax.annotate(f"{ratio}x", xy=(2, _gpt2_rtn), xytext=(1.5, 3000),
            fontsize=14, fontweight="bold", color=C_RED,
            arrowprops=dict(arrowstyle="->", color=C_RED, lw=2))

ax.set_ylim(10, 50000)

fig.savefig("figures/gptq_vs_rtn.png")
plt.show()
plt.close(fig)
generated.append("gptq_vs_rtn.png")

# =====================================================================
# 4. extreme_quantization.png (known data — requires multi-bit runs)
# =====================================================================
bits = [4, 3, 2]
gptq_ppl = [35.45, 765.03, 11749.80]
rtn_ppl = [14434, 8559.39, 51045.48]
fp16_val = 25.17

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(bits, gptq_ppl, "o-", color=C_BLUE, linewidth=2.5, markersize=9,
        label="GPTQ", zorder=3)
ax.plot(bits, rtn_ppl, "s-", color=C_RED, linewidth=2.5, markersize=9,
        label="Naive RTN", zorder=3)
ax.axhline(y=fp16_val, color=C_GREY, linestyle="--", linewidth=1.5,
           label=f"FP16 baseline ({fp16_val})", zorder=2)

ax.set_yscale("log")
ax.set_xlabel("Bit-Width", fontsize=12)
ax.set_ylabel("WikiText-2 Perplexity (log scale)", fontsize=12)
ax.set_xticks(bits)
ax.set_xticklabels(["4-bit", "3-bit", "2-bit"])
ax.invert_xaxis()
ax.legend(loc="upper right", framealpha=0.9, fontsize=11)
ax.set_title("GPT-2: Perplexity vs Bit-Width", fontsize=14, fontweight="bold")
ax.yaxis.grid(True, alpha=0.3)

for b, v in zip(bits, gptq_ppl):
    ax.annotate(f"{v:.0f}" if v > 100 else f"{v:.1f}",
                xy=(b, v), xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=9, color=C_BLUE, fontweight="bold")

fig.savefig("figures/extreme_quantization.png")
plt.show()
plt.close(fig)
generated.append("extreme_quantization.png")

# =====================================================================
# 5. algorithm_diagram.png
# =====================================================================
fig, ax = plt.subplots(figsize=(12, 3))
ax.set_xlim(0, 10)
ax.set_ylim(0, 2)
ax.axis("off")

boxes = [
    (0.3,  "Calibration Data\n(128 samples)"),
    (2.3,  "Hessian\n$H = X^\\top X / n$"),
    (4.3,  "Cholesky\n$H^{-1}$"),
    (6.3,  "Column-by-column\nquantize + compensate"),
    (8.5,  "Quantized\nWeights"),
]

box_w = 1.7
box_h = 1.2
box_y = 0.4
box_colors = [C_GREY, C_ORANGE, C_ORANGE, C_BLUE, C_GREEN]

for i, (bx, text) in enumerate(boxes):
    rect = mpatches.FancyBboxPatch(
        (bx, box_y), box_w, box_h,
        boxstyle="round,pad=0.12",
        facecolor=box_colors[i], alpha=0.18,
        edgecolor=box_colors[i], linewidth=2,
    )
    ax.add_patch(rect)
    ax.text(bx + box_w / 2, box_y + box_h / 2, text,
            ha="center", va="center", fontsize=10, fontweight="bold",
            color="#2c3e50")

    if i < len(boxes) - 1:
        next_bx = boxes[i + 1][0]
        ax.annotate("", xy=(next_bx - 0.05, box_y + box_h / 2),
                    xytext=(bx + box_w + 0.05, box_y + box_h / 2),
                    arrowprops=dict(arrowstyle="-|>", color="#7f8c8d", lw=2))

ax.set_title("GPTQ Algorithm Pipeline", fontsize=14, fontweight="bold", pad=15)

fig.savefig("figures/algorithm_diagram.png")
plt.show()
plt.close(fig)
generated.append("algorithm_diagram.png")

print(f"\nGenerated {len(generated)} figures: {', '.join(generated)}")
